# NHANES: data and model evaluation

A reproducible walkthrough of the actual supplied files. This notebook does not train a separate model or expose participant IDs. Read `docs/METHODOLOGY.md` before interpreting estimates. Use the same Python 3.12 environment as training.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from nhanes_ai.data import read_cycles, prepare_cohort
from nhanes_ai.schema import LabelConfig

ROOT = Path.cwd()
if not (ROOT / 'DATA_SET').exists():
    ROOT = ROOT.parent
assert (ROOT / 'DATA_SET').is_dir()

## Validate joins and audit filtering

Every module joins one-to-one within its survey cycle. Unknown disease evidence stays unknown. The function prints the remaining count after sequential exclusions.

In [ ]:
cohort, audit = prepare_cohort(read_cycles(ROOT / 'DATA_SET'), LabelConfig())
display(pd.Series(audit['exclusions'], name='Excluded records'))
display(cohort.groupby('cycle').size().rename('Remaining records').to_frame())
assert audit['remaining_records'] == 10623

## Eight inputs and three shared derived measurements
Family history follows MCQ300C (diabetes) and MCQ300A (heart attack/angina before 50 in parents/siblings). Refused/unknown is not no. BMI and both masses are recalculated from weight, height and fat percentage, as in the app.


In [ ]:
from nhanes_ai.schema import USER_FIELDS, FEATURES
from nhanes_ai.features import derive_features
display(pd.DataFrame(audit["family_history"]).T)
assert len(USER_FIELDS) == 8 and len(FEATURES) == 11
assert "ethnicity" not in FEATURES and "bone_mass_kg" not in FEATURES
pd.testing.assert_frame_equal(derive_features(cohort[USER_FIELDS]), cohort[FEATURES])


## Target availability and imbalance

These are aggregate counts, not survey-weighted prevalence estimates. The cardiac outcomes are rare in this DXA-eligible cohort, which excludes adults aged 60 and older.

In [ ]:
counts = pd.DataFrame(audit['targets']).T
counts['observed_positive_fraction'] = counts['positive'] / counts['known']
display(counts)
fig = px.bar(counts.reset_index(), x='index', y='positive', labels={'index': 'Study target', 'positive': 'Observed positive cases'})
fig.show(renderer='plotly_mimetype')

## Compare all fitted candidates

Run `python -m nhanes_ai.train --jobs 2` from the project root first. This is the 2015–2016 validation comparison; it is not the final test score. Models are selected by average precision, not accuracy.

In [ ]:
comparison = pd.read_csv(ROOT / 'reports/model_comparison.csv')
assert len(comparison) == 32
display(comparison[['disease','algorithm','recall','precision','f1','roc_auc','pr_auc']])

## Temporal 2017–2018 results

Do not change thresholds using this table. Low precision or recall cannot be repaired by reporting ROC-AUC alone. Participant-bootstrap intervals and subgroup metrics are in the JSON report.
This test cycle was inspected in version 1; these are version-comparison results, not new external validation.


In [ ]:
results = json.loads((ROOT / 'reports/test_metrics.json').read_text())
summary = pd.DataFrame({disease: {key: value[key] for key in ['algorithm','positive','n','recall','precision','f1','roc_auc','pr_auc','brier']} for disease,value in results.items()}).T
display(summary)

## Limits

These are cross-sectional associations in US survey participants with completed DXA exams, not future disease risk or clinical diagnoses. SHAP explains model behavior and is not causal. Ethnicity is excluded from predictors; family-history unknowns stay unknown. The Arabic interface does not establish validity in non-US populations. See the generated model card for observed failures and deployment boundaries.